#### Raw data from yahoo finance

In [ ]:
import yfinance as yf


def download_and_process_data(tickers, output_file):
    # Download data for the given tickers
    data = yf.download(
        tickers,
        start=None,
        period="max",
        interval="1d",
        auto_adjust=False,
        actions=False,
        progress=False,
    )

    # Process the data
    data.columns = data.columns.droplevel("Ticker")
    df = data.reset_index()
    df = df.sort_values("Date", ascending=False)
    df["Date"] = df["Date"].dt.strftime("%b %d, %Y")

    df = df[["Date", "Open", "High", "Low", "Close", "Adj Close"]]

    # Rename column and format numeric columns
    df.rename(columns={"Adj Close": "Adj Close**"}, inplace=True)
    for col in ["Open", "High", "Low", "Close", "Adj Close**"]:
        df[col] = df[col].apply(lambda x: f"{x:,.2f}")

    # Save to a CSV file
    df.to_csv(output_file, index=False)


# Define tickers and corresponding output files
assets = {
    "^GSPC": "sp500_raw_updated.csv",
    "^SP500TR": "sp500div_raw_updated.csv",
    "SPY": "spy_raw_updated.csv",
    "ULPIX": "ulpix_raw_updated.csv",
    "UPRO": "upro_raw_updated.csv",
    "EURUSD=X": "eurusd_raw_updated.csv",
    "SXR8.DE": "sxr8.de_raw_updated.csv",
    "DBPG.DE": "dbpg.de_raw_updated.csv",
    "3USL.MI": "3usl.mi_raw_updated.csv",  # broken due to weird change dec 19 2022
    "^NDX": "nasdaq100_raw_updated.csv",
    "QQQ": "qqq_raw_updated.csv",
    "QLD": "qld_raw_updated.csv",
    "TQQQ": "tqqq_raw_updated.csv",
    "EQQQ.MI": "eqqq.mi_raw_updated.csv",
    "L8I7.DE": "l8i7.mi_raw_updated.csv",
    "QQQ3.MI": "qqq3.mi_raw_updated.csv",
}

# Loop through assets and process
for ticker, file_name in assets.items():
    download_and_process_data([ticker], file_name)


## Since LIBOR is not availabele we need to use SOFR and merge the series
First we find a good date to switch the series

In [ ]:
import requests
from datetime import datetime
import pandas as pd
from io import BytesIO

params = {
    "startDt": "2000-04-01",  # Adjust as needed
    "endDt": datetime.now().strftime("%Y-%m-%d"),  # Current date in YYYY-MM-DD format
    "productCode": "50",  # SOFR product code
    "format": "xlsx",  # Request Excel format
}

url = "https://markets.newyorkfed.org/read"
response = requests.get(url, params=params)

if response.status_code == 200:
    # Read the Excel data into a pandas DataFrame
    sofr = pd.read_excel(BytesIO(response.content))
else:
    print(f"Error fetching data: HTTP {response.status_code}")

In [ ]:
import pandas as pd

libor = pd.read_csv("../raw_data/USDONTD156N.csv", na_values=["."])
libor["DATE"] = pd.to_datetime(libor["DATE"])
libor = libor.set_index("DATE")
libor = libor.resample("D").ffill()
libor = libor.bfill()
libor = libor.reset_index()

sofr = pd.read_excel("./Secured Overnight Financing Rate.xlsx")
# https://markets.newyorkfed.org/read?startDt=2000-04-01&endDt=2025-04-14&eventCodes=520&productCode=50&sort=postDt:-1,eventCode:1&format=xlsx
sofr = sofr[sofr["Rate Type"] == "SOFR"]
sofr = sofr[["Effective Date", "Rate (%)"]]
sofr = sofr.rename(columns={"Effective Date": "DATE", "Rate (%)": "USDONTD156N"})
sofr["DATE"] = pd.to_datetime(sofr["DATE"])
sofr = sofr.set_index("DATE")
sofr = sofr.resample("D").ffill()
sofr = sofr.bfill()
sofr = sofr.reset_index()
sofr = sofr.reset_index()

first_overlap_date = max(libor["DATE"].min(), sofr["DATE"].min())
last_overlap_date = min(libor["DATE"].max(), sofr["DATE"].max())


libor_filtered = libor[
    (libor["DATE"] >= first_overlap_date) & (libor["DATE"] <= last_overlap_date)
]
sofr_filtered = sofr[
    (sofr["DATE"] >= first_overlap_date) & (sofr["DATE"] <= last_overlap_date)
]

In [ ]:
import matplotlib.pyplot as plt

merged = pd.merge(
    libor_filtered, sofr_filtered, on="DATE", suffixes=("_libor", "_sofr")
)
merged = merged.drop(columns=["index"])

merged["squared_diff"] = (merged["USDONTD156N_libor"] - merged["USDONTD156N_sofr"]) ** 2

window_size = 30
merged["mse"] = merged["squared_diff"].rolling(window=window_size).mean()

best_match_idx = merged["mse"].idxmin()
best_match_date = merged.loc[best_match_idx, "DATE"]
print(f"Most similar point in time series: {best_match_date}")

### Merge SOFR and LIBOR

In [ ]:
libor_up_to_match = libor[libor["DATE"] <= best_match_date]
sofr_after_match = sofr[sofr["DATE"] > best_match_date]
final_df = pd.concat([libor_up_to_match, sofr_after_match]).reset_index(drop=True)
final_df = final_df.drop(columns=["index"])

plt.figure(figsize=(12, 6))
plt.plot(libor["DATE"], libor["USDONTD156N"], label="LIBOR", alpha=0.6)
plt.plot(sofr["DATE"], sofr["USDONTD156N"], label="SOFR", alpha=0.6)
plt.plot(
    final_df["DATE"],
    final_df["USDONTD156N"],
    label="Final Transition",
    color="black",
    linewidth=2,
)
plt.axvline(x=best_match_date, color="red", linestyle="--", label="Transition Point")
plt.legend()
plt.xlabel("Date")
plt.ylabel("Value")
plt.title("LIBOR to SOFR Transition")
plt.show()

In [ ]:
final_df.to_csv("USDONTD156N_updated.csv", index=False)